# Getting Started with the MoistX API

MoistX provides satellite-derived **surface soil moisture** (0–5 cm depth) fusing data from
**Sentinel-1**, **Sentinel-2**, and **Landsat 8/9**.

In this notebook you will:
- Authenticate with the MoistX API using an API key
- Query soil moisture for a point in the **Thessaly agricultural plain**, Greece
- Parse the JSON response into a pandas DataFrame
- Build an interactive time series coloured by satellite sensor


## Prerequisites

Install dependencies (run once):
```bash
pip install requests pandas plotly
```

Get your API key from [moistx.com/dashboard](https://moistx.com/dashboard) and either:
- Set the environment variable `MOISTX_API_KEY=your_key`, **or**
- Replace `'your_api_key_here'` in the configuration cell below.

Full endpoint reference: [Swagger UI](https://moistx.com/api/docs) /
[ReDoc](https://moistx.com/api/redoc). Data is free to use/share/modify (including
commercially) with attribution to MoistX and the underlying satellite sources — see the
repo [README](./README.md#data-license--attribution) for the exact wording.


In [ ]:
import os
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE_URL = 'https://moistx.com/api'
API_KEY  = os.getenv('MOISTX_API_KEY', 'your_api_key_here')
HEADERS  = {'X-Api-Key': API_KEY}


## Location and time range

We query a reference point in the **Thessaly plain** (central Greece), one of the most
intensively farmed agricultural regions in the country.

The API accepts:
- `latitude` / `longitude` — decimal degrees, WGS84
- `start_datetime` / `end_datetime` — ISO 8601 format (`YYYY-MM-DDTHH:MM:SS`)
- Maximum window: **365 days** per request


In [2]:
LATITUDE  = 39.55
LONGITUDE = 22.30
START     = '2024-01-01T00:00:00'
END       = '2024-12-31T00:00:00'


## Fetching soil moisture data


In [3]:
response = requests.get(
    f'{BASE_URL}/get/point/soil-moisture',
    headers=HEADERS,
    params={
        'latitude':       LATITUDE,
        'longitude':      LONGITUDE,
        'start_datetime': START,
        'end_datetime':   END,
    },
)
response.raise_for_status()
data = response.json()
print(f'Received {len(data)} observations')


Received 146 observations


## Understanding the response

Each entry in the JSON response maps a timestamp to a dictionary with:
- `sm_0_5cm` — volumetric soil moisture at 0–5 cm depth (vol%)
- `source` — the satellite/data source that acquired the image: `S2`, `S1_asc`, `S1_desc`, `L89`, or `harmonized` (a combined estimate from multiple sensors, used by some regions)


In [4]:
sample_key = next(iter(data))
entry = data[sample_key]
print('Timestamp :', sample_key)
print('  sm_0_5cm:', entry['sm_0_5cm'], 'vol%')
print('  source  :', entry['source'])


Timestamp : 2024-01-04T09:30:00
  sm_0_5cm: 28.7 vol%
  source  : harmonized


## Building a DataFrame


In [5]:
records = [
    {'datetime': pd.to_datetime(ts), 'soil_moisture': v['sm_0_5cm'], 'source': v['source']}
    for ts, v in data.items()
]
df = pd.DataFrame(records).set_index('datetime').sort_index()

print('Date range :', df.index.min().date(), 'to', df.index.max().date())
print('Sources    :', sorted(df['source'].unique()))
df.head()


Date range : 2024-01-04 to 2024-11-24
Sources    : ['harmonized']


,soil_moisture,source
datetime,,
2024-01-04 09:30:00,28.7,harmonized
2024-01-05 09:30:00,31.2,harmonized
2024-01-08 09:30:00,30.4,harmonized
2024-01-10 09:30:00,32.9,harmonized
2024-01-11 09:30:00,30.7,harmonized


## Interactive time series

Each dot represents one satellite acquisition. Colour indicates the sensor.
Hover over any point to see the exact date and soil moisture value.


In [6]:
SOURCE_COLORS = {
    'S2':      '#4CAF50',
    'S1_asc':  '#2196F3',
    'S1_desc': '#7B1FA2',
    'L89':     '#FF9800',
}

fig = px.scatter(
    df.reset_index(),
    x='datetime',
    y='soil_moisture',
    color='source',
    color_discrete_map=SOURCE_COLORS,
    title='Surface Soil Moisture (0–5 cm) — Thessaly Plain, 2024',
    labels={
        'soil_moisture': 'Soil Moisture (vol%)',
        'datetime': 'Date',
        'source': 'Satellite',
    },
)
fig.update_traces(marker_size=8)
fig.update_layout(height=450, template='plotly_white', hovermode='x unified')
fig.show()


## Source coverage summary


In [7]:
summary = df.groupby('source').agg(
    observations=('soil_moisture', 'count'),
    mean_sm=('soil_moisture', 'mean'),
    min_sm=('soil_moisture', 'min'),
    max_sm=('soil_moisture', 'max'),
).round(1)
summary


,observations,mean_sm,min_sm,max_sm
source,,,,
harmonized,146,20.8,9.5,32.9
